# Final PCA: per-sample-set covariate PCs

`01_ancestry_pca_filter.ipynb` decides membership (AoU's own premade PCs). This notebook only produces the covariate PCs `02_residualize_phenotypes.ipynb` reads.

Refits rather than reusing AoU's premade PCs directly -- those are fit globally across all ancestries, so PC1/PC2 there mostly separate continental groups rather than resolving structure within one final `SAMPLE_SET`. Reads `03_genome_wide_qc_thinning_batch_submit.ipynb`'s GRM panel for this `SAMPLE_SET` (already restricted to its final members -- no `--keep` needed here), restricts further to just the HM3 portion, excludes known long-range high-LD regions, r2-prunes, and fits `--pca approx` to `N_PCS=10`. Also projects 1000G reference samples onto the resulting PC space for orientation.

## Compute resource

8-16 vCPU is plenty -- runs on the thinned subset, not the full GRM panel.

## Setup

plink2: manual install, same pattern as everywhere else in this repo.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version
nproc
free -h

In [ ]:
import os
import pandas as pd

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

N_THREADS = os.cpu_count()

## Sample set configuration

`prob_tag` must match `01_ancestry_pca_filter.ipynb`'s `SAMPLE_SETS` dict -- that's what named the keep-list files.

In [ ]:
SAMPLE_SETS = {
    "eur_strict": {"prob_tag": "p5"},
    "eur_base":   {"prob_tag": "p10"},
    "eur_loose":  {"prob_tag": "p50"},
    "uniform": {"prob_tag": "uniform"},
    "afr":        {"prob_tag": "p1"},
    "eas":        {"prob_tag": "p50"},
}

N_PCS = 10              # final covariate PC count

CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR = "phenotypic_covariance_v9"
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

KG_DIR = f"{WORKSPACE_BUCKET}/1000g_reference"   # explore_1kg_reference.ipynb's output, CDR-independent
KG_OUT_PREFIX = f"{KG_DIR}/1kg_all_qc"           # HM3-restricted, QC'd 1000G reference bfile
KG_PANEL_PATH = f"{KG_DIR}/integrated_call_samples_v3.20130502.ALL.panel"   # sample/pop/super_pop labels
assert os.path.isfile(f"{KG_OUT_PREFIX}.acount"), f"missing {KG_OUT_PREFIX}.acount -- run explore_1kg_reference.ipynb first"

## Inputs

Run once per `SAMPLE_SET`. Copies that `SAMPLE_SET`'s own GRM panel locally -- already restricted to its final members, no `--keep` needed.

In [ ]:
SAMPLE_SET = "eur_base"   # <-- change this and rerun for each of the 6 sample sets
_cfg = SAMPLE_SETS[SAMPLE_SET]

PANEL_DIR = f"{ANCESTRY_BUCKET_DIR}/genome_wide_panel_{SAMPLE_SET}"
MERGED_NAME = f"genome_wide_thinned_{CDR_VERSION}_{SAMPLE_SET}"   # pgen form, from 03_genome_wide_qc_thinning_batch_submit.ipynb's merge step
MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, MERGED_NAME)

for ext in ("pgen", "pvar", "psam"):
    bucket_path = f"{PANEL_DIR}/{MERGED_NAME}.{ext}"
    local_path = f"{MERGED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), (
        f"missing GRM panel: {bucket_path!r} -- run 03_genome_wide_qc_thinning_batch_submit.ipynb's merge section first"
    )
    if not os.path.isfile(local_path) or os.path.getsize(local_path) != os.path.getsize(bucket_path):
        import shutil
        shutil.copy(bucket_path, local_path)

PCA_BED_PREFIX = os.path.join(LOCAL_WORK_DIR, f"{MERGED_NAME}_pca_thinned")   # per-SAMPLE_SET now (separate GRM panel each)

FINAL_PCA_BUCKET_DIR = f"{ANCESTRY_BUCKET_DIR}/ancestry_pca_filter/final_pca"
SAMPLE_SET_KEEP_PATH = os.path.join(FINAL_PCA_BUCKET_DIR, SAMPLE_SET, f"final_keep_ids_{SAMPLE_SET}_{_cfg['prob_tag']}.txt")
assert os.path.isfile(SAMPLE_SET_KEEP_PATH), (
    f"missing keep-list: {SAMPLE_SET_KEEP_PATH!r} -- run 01_ancestry_pca_filter.ipynb first"
)

SAMPLE_SET_OUT_DIR = os.path.join(FINAL_PCA_BUCKET_DIR, SAMPLE_SET)
os.makedirs(SAMPLE_SET_OUT_DIR, exist_ok=True)
FINAL_PCA_PREFIX = os.path.join(LOCAL_WORK_DIR, f"final_pca_{CDR_VERSION}_{SAMPLE_SET}")

print(MERGED_PREFIX)
print(SAMPLE_SET_KEEP_PATH)
print(SAMPLE_SET_OUT_DIR)

## Restrict to HM3 before pruning

`MERGED_PREFIX` (this `SAMPLE_SET`'s GRM panel) is HM3 union randomly-thinned SNPs, both already `--keep`+QC'd (`02_build_unified_panel.ipynb`/`03_genome_wide_qc_thinning_batch_submit.ipynb`). Real ID+REF+ALT harmonization against `explore_1kg_reference.ipynb`'s `1kg_all_qc.acount` (not a bare ID match) recovers just the HM3 portion that survived this `SAMPLE_SET`'s own QC -- the standard, curated reference set for ancestry PCA, already excludes the classic MHC list, and a much better PCA input than the panel's random-thin component.

In [ ]:
%%bash -s "$MERGED_PREFIX" "$KG_OUT_PREFIX"
set -e
MERGED_PREFIX=$1
KG_OUT_PREFIX=$2

grep -v '^##' "${MERGED_PREFIX}.pvar" | awk 'NR>1 {print $3, $4, $5}' | LC_ALL=C sort > "${MERGED_PREFIX}_id_ref_alt.sorted"
awk 'NR>1 {print $2, $3, $4}' "${KG_OUT_PREFIX}.acount" | LC_ALL=C sort > "${MERGED_PREFIX}_kg_id_ref_alt.sorted"
LC_ALL=C comm -12 "${MERGED_PREFIX}_id_ref_alt.sorted" "${MERGED_PREFIX}_kg_id_ref_alt.sorted" | awk '{print $1}' > "${MERGED_PREFIX}_hm3_agreeing.ids"

echo "HM3-agreeing within this SAMPLE_SET's GRM panel: $(wc -l < "${MERGED_PREFIX}_hm3_agreeing.ids")"

## Known long-range high-LD regions (GRCh38)

`--indep-pairwise`'s 50-variant window (~100-150kb at this panel's density) is too narrow to catch long-range LD -- known blocks like the HLA locus, the chr8/chr17 inversions, and the LCT region span megabases and routinely dominate the top few PCs even after standard short-window pruning. Excluded explicitly before pruning, same fix used throughout genotype-PCA QC pipelines generally.

GRCh38 coordinates from the `plinkQC` package's `high-LD-regions-hg38-GRCh38.txt` (github.com/meyer-lab-cshl/plinkQC) -- plink-ready range format (`CHR START END SETID`), embedded here rather than fetched live so this doesn't depend on GitHub staying up. Assumes this panel's `CHROM` values are `chr1`..`chr22` (AoU/ACAF's GRCh38 convention) -- confirm against a real `.pvar` if these regions don't visibly clean up the loadings plot.

In [ ]:
HIGH_LD_REGIONS_GRCH38 = """\
chr1 47761740 51761740 1
chr2 85919365 100517106 2
chr2 89917298 89917322 3
chr2 87416141 87416186 4
chr2 87417804 87417863 5
chr2 87418924 87418981 6
chr1 144106678 144106709 7
chr14 87391719 87391996 8
chr9 40365644 40365693 9
chr2 182427027 189427029 10
chr3 47483505 49987563 11
chr3 83368158 86868160 12
chr5 44464140 51168409 13
chr5 129636407 132636409 14
chr6 25391792 33424245 15
chr6 26726947 26726981 16
chr6 57788603 58453888 17
chr6 61109122 61357029 18
chr6 61424410 61424451 19
chr9 64198500 64200392 20
chr6 139637169 142137170 21
chr7 54964812 66897578 22
chr7 62182500 62277073 23
chr12 34639034 34639084 24
chr14 94658026 94658080 25
chr17 43159541 43159574 26
chr2 135275091 135275210 27
chr1 181955019 181955047 28
chr22 30060084 30060162 29
chr9 88958735 88959017 30
chr2 207609786 207609808 31
chr22 42980497 42980522 32
chr20 4031884 4032441 33
chr8 8105067 12105082 34
chr8 43025699 48924888 35
chr8 47303500 47317337 36
chr8 110918594 113918595 37
chr10 36671065 43184546 38
chr10 41693521 41885273 39
chr1 125169943 125170022 40
chr11 88127183 91127184 41
chr12 32955798 41319931 42
chr20 33948532 36438183 43
"""

LD_REGIONS_PATH = os.path.join(LOCAL_WORK_DIR, "high_ld_regions_grch38.txt")
with open(LD_REGIONS_PATH, "w") as f:
    f.write(HIGH_LD_REGIONS_GRCH38)
print(f"Wrote {LD_REGIONS_PATH}")

## Prune HM3, no further thinning

`--extract` HM3-agreeing IDs, `--exclude range` the known long-range high-LD regions above, `--indep-pairwise 50 10 0.1` (same window/step/r2 `explore_1kg_reference.ipynb` uses). Output is the final PCA variant set directly -- **no `--thin`-to-target step anymore**: HM3 is already the standard, appropriately-sized reference set for ancestry PCA, and randomly thinning it further was only ever a speed hack for the old, much larger GRM-panel-based input. Not shared/cached across `SAMPLE_SET`s -- each has its own separate GRM panel.

In [ ]:
%%bash -s "$MERGED_PREFIX" "$PCA_BED_PREFIX" "$N_THREADS" "$LD_REGIONS_PATH"
set -e
MERGED_PREFIX=$1
PCA_BED_PREFIX=$2
THREADS=$3
LD_REGIONS_PATH=$4

if [ -s "${PCA_BED_PREFIX}.pgen" ]; then
  echo "already pruned, skipping"
else
  PRUNE_PREFIX="${MERGED_PREFIX}_pruned"
  plink2 \
    --pfile "$MERGED_PREFIX" \
    --extract "${MERGED_PREFIX}_hm3_agreeing.ids" \
    --exclude range "$LD_REGIONS_PATH" \
    --nonfounders \
    --indep-pairwise 50 10 0.1 \
    --threads "$THREADS" \
    --out "$PRUNE_PREFIX"

  echo "Pruned HM3 SNPs: $(wc -l < "${PRUNE_PREFIX}.prune.in")"

  plink2 \
    --pfile "$MERGED_PREFIX" \
    --extract "${PRUNE_PREFIX}.prune.in" \
    --threads "$THREADS" \
    --make-pgen \
    --out "$PCA_BED_PREFIX"
fi

echo "Final PCA SNP count:"
awk 'END{print NR-1}' "${PCA_BED_PREFIX}.pvar"

In [ ]:
%%bash -s "$PCA_BED_PREFIX" "$FINAL_PCA_PREFIX" "$N_THREADS" "$N_PCS"
set -e
PCA_BED_PREFIX=$1
FINAL_PCA_PREFIX=$2
THREADS=$3
NPCS=$4

plink2 \
  --pfile "$PCA_BED_PREFIX" \
  --nonfounders \
  --freq counts \
  --pca approx "$NPCS" allele-wts \
  --threads "$THREADS" \
  --out "$FINAL_PCA_PREFIX"

ls -lh "${FINAL_PCA_PREFIX}".*

### Scree plot

% variance explained per PC, within this `SAMPLE_SET`.

In [ ]:
import matplotlib.pyplot as plt

eigenval = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenval", header=None, names=["eigenvalue"])
eigenval["pc"] = range(1, len(eigenval) + 1)
eigenval["pct_variance"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(eigenval["pc"], eigenval["pct_variance"], color="royalblue")
ax.set_xlabel("PC")
ax.set_ylabel("% variance explained")
ax.set_title(f"Final PCA [{SAMPLE_SET}] scree plot")
ax.set_xticks(eigenval["pc"])
plt.tight_layout()
plot_path = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_scree_{SAMPLE_SET}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")
print(eigenval[["pc", "eigenvalue", "pct_variance"]].to_string(index=False))

## Write PC covariates for 02_residualize_phenotypes.ipynb

`IID PC1 ... PC10` format.

In [ ]:
direct = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenvec", sep=r"\s+")

_id_col = "#IID" if "#IID" in direct.columns else "IID"
pc_cols = [c for c in direct.columns if c.startswith("PC")]
assert len(pc_cols) == N_PCS, f"expected {N_PCS} PC columns, found {len(pc_cols)}: {pc_cols}"

covariate_table = direct[[_id_col] + pc_cols].rename(columns={_id_col: "IID"})
covariate_table["IID"] = covariate_table["IID"].astype(str)

PC_COVARIATE_PATH = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_pc_covariates_{SAMPLE_SET}.txt")
covariate_table.to_csv(PC_COVARIATE_PATH, sep="\t", index=False)
print(f"Wrote {len(covariate_table)} samples' PC1-PC{N_PCS} -> {PC_COVARIATE_PATH}")

## Write PCA loadings

`allele-wts` output (`.eigenvec.allele`) -- `ID`, `A1`, `PC1`..PC{N_PCS} allele weights. Needed to project new samples onto this exact `SAMPLE_SET`'s PC space later (e.g. out-of-sample projection), not read by `02_residualize_phenotypes.ipynb`.

In [ ]:
import shutil

LOADINGS_SRC = f"{FINAL_PCA_PREFIX}.eigenvec.allele"
LOADINGS_PATH = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_loadings_{SAMPLE_SET}.txt")
shutil.copy(LOADINGS_SRC, LOADINGS_PATH)
print(f"Wrote loadings -> {LOADINGS_PATH}")

## Plot loadings (Manhattan-style)

Loading magnitude by genomic position, first 4 PCs -- the standard check for whether a PC is driven by a handful of variants in one region (an inversion, HLA, LCT, an under-pruned high-LD block) rather than genome-wide structure. IDs are `chrom:pos:ref:alt` (`--set-all-var-ids` in `03_genome_wide_qc_thinning_batch_submit.ipynb`), so CHROM/POS parse straight out of the ID column.

In [ ]:
N_PCS_TO_PLOT = 4

loadings = pd.read_csv(LOADINGS_PATH, sep=r"\s+")
id_col = "#ID" if "#ID" in loadings.columns else "ID"
loadings[["CHROM", "POS"]] = loadings[id_col].str.split(":", n=2, expand=True).iloc[:, :2]
loadings["CHROM"] = loadings["CHROM"].astype(str)
loadings["POS"] = loadings["POS"].astype(int)

chrom_order = [str(c) for c in range(1, 23)]
loadings = loadings[loadings["CHROM"].isin(chrom_order)].copy()
loadings["CHROM"] = pd.Categorical(loadings["CHROM"], categories=chrom_order, ordered=True)
loadings = loadings.sort_values(["CHROM", "POS"])

# cumulative x-position across chromosomes, standard Manhattan-plot construction
chrom_offsets = {}
offset = 0
chrom_centers = {}
for chrom in chrom_order:
    chrom_offsets[chrom] = offset
    chrom_max = loadings.loc[loadings["CHROM"] == chrom, "POS"].max()
    if pd.notna(chrom_max):
        chrom_centers[chrom] = offset + chrom_max / 2
        offset += chrom_max
loadings["CUM_POS"] = loadings.apply(lambda r: r["POS"] + chrom_offsets[r["CHROM"]], axis=1)

pc_cols = [f"PC{i}" for i in range(1, N_PCS_TO_PLOT + 1)]
colors = ["tab:blue", "tab:orange"]

fig, axes = plt.subplots(len(pc_cols), 1, figsize=(14, 3 * len(pc_cols)), sharex=True)
for ax, pc in zip(axes, pc_cols):
    for i, chrom in enumerate(chrom_order):
        sub = loadings[loadings["CHROM"] == chrom]
        ax.scatter(sub["CUM_POS"], sub[pc] ** 2, s=2, alpha=0.5, color=colors[i % 2])
    ax.set_ylabel(f"{pc} loading²")

axes[-1].set_xticks([chrom_centers[c] for c in chrom_order if c in chrom_centers])
axes[-1].set_xticklabels(chrom_order, fontsize=7)
axes[-1].set_xlabel("Chromosome")
fig.suptitle(f"Final PCA [{SAMPLE_SET}] loadings by genomic position")
plt.tight_layout()
plot_path = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_loadings_manhattan_{SAMPLE_SET}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Project 1000G onto this `SAMPLE_SET`'s PC space

Reverse direction of `explore_1kg_projection_crosscheck.ipynb` (which projects AoU onto 1000G's own PCA): fit is already this `SAMPLE_SET`'s own, so instead project `explore_1kg_reference.ipynb`'s 1000G reference samples onto it, using the loadings just written above. Shows where reference populations (CEU, YRI, ...) land relative to the exact PCs used downstream -- the more directly interpretable orientation for these particular PCs than checking against 1000G's own space.

In [ ]:
%%bash -s "$PCA_BED_PREFIX" "$KG_OUT_PREFIX" "$FINAL_PCA_PREFIX" "$N_PCS" "$LOCAL_WORK_DIR" "$SAMPLE_SET"
set -e
PCA_BED_PREFIX=$1
KG_OUT_PREFIX=$2
FINAL_PCA_PREFIX=$3
NPCS=$4
LOCAL_WORK_DIR=$5
SAMPLE_SET=$6

PROJECT_OUT="${LOCAL_WORK_DIR}/1kg_projected_onto_${SAMPLE_SET}"

grep -v '^##' "${PCA_BED_PREFIX}.pvar" | awk 'NR>1 {print $3, $4, $5}' | LC_ALL=C sort > "${PROJECT_OUT}_aou_id_ref_alt.sorted"
awk 'NR>1 {print $2, $3, $4}' "${KG_OUT_PREFIX}.acount" | LC_ALL=C sort > "${PROJECT_OUT}_kg_id_ref_alt.sorted"
LC_ALL=C comm -12 "${PROJECT_OUT}_aou_id_ref_alt.sorted" "${PROJECT_OUT}_kg_id_ref_alt.sorted" | awk '{print $1}' > "${PROJECT_OUT}_agreeing.ids"
echo "Agreeing with 1000G for projection: $(wc -l < "${PROJECT_OUT}_agreeing.ids")"

WEIGHTS="${FINAL_PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx "PC${NPCS}" | cut -d: -f1)

plink2 \
  --bfile "$KG_OUT_PREFIX" \
  --extract "${PROJECT_OUT}_agreeing.ids" \
  --nonfounders \
  --read-freq "${FINAL_PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$PROJECT_OUT"

## Plot: this `SAMPLE_SET`'s AoU samples vs. 1000G reference populations

PC1 vs PC2, in this `SAMPLE_SET`'s own final PC space. Also persists the 1000G projected scores for later use.

In [ ]:
kg_projected = pd.read_csv(f"{LOCAL_WORK_DIR}/1kg_projected_onto_{SAMPLE_SET}.sscore", sep=r"\s+")
kg_id_col = "#IID" if "#IID" in kg_projected.columns else "IID"
kg_projected = kg_projected.rename(columns={kg_id_col: "sample"})

kg_panel = pd.read_csv(KG_PANEL_PATH, sep="\t")[["sample", "pop", "super_pop"]]
kg_projected = kg_projected.merge(kg_panel, on="sample", how="left")

KG_PROJECTION_PATH = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_1kg_projection_{SAMPLE_SET}.txt")
kg_projected.to_csv(KG_PROJECTION_PATH, sep="\t", index=False)
print(f"Wrote {len(kg_projected)} projected 1000G samples -> {KG_PROJECTION_PATH}")

aou_pcs = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenvec", sep=r"\s+")

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(aou_pcs["PC1"], aou_pcs["PC2"], s=3, alpha=0.15, color="lightgray", label=f"AoU [{SAMPLE_SET}]")
for pop, sub in kg_projected.groupby("super_pop"):
    ax.scatter(sub["PC1_AVG"], sub["PC2_AVG"], s=14, marker="x", label=f"1000G: {pop}")

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(f"Final PCA [{SAMPLE_SET}] vs. 1000G reference populations")
ax.legend(fontsize=7, markerscale=1.5, loc="best")
plt.tight_layout()
plot_path = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_1kg_overlay_{SAMPLE_SET}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Next steps

`02_residualize_phenotypes.ipynb` already points at these output paths -- no manual copying needed.